In [30]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)
y = y.astype(int)

X = X / 255.0

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y)

classifier = MLPClassifier()
classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)

print(f"MLP Accuracy: {accuracy_score(y_test, y_pred)}")
print(f"F1 Micro: {f1_score(y_test, y_pred, average='micro')}")
print(f"F1 Macro: {f1_score(y_test, y_pred, average='macro')}")

MLP Accuracy: 0.9773714285714286
F1 Micro: 0.9773714285714286
F1 Macro: 0.977174159820993


In [31]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST('./data', train=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000)

In [32]:
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.max_pool2d(x, 2)
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

In [33]:
import torch.optim as optim

model = CNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [34]:
epochs = 10

for epoch in range(epochs):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.4f}")

Epoch 1/10, Loss: 0.1868
Epoch 2/10, Loss: 0.0051
Epoch 3/10, Loss: 0.0188
Epoch 4/10, Loss: 0.0074
Epoch 5/10, Loss: 0.0029
Epoch 6/10, Loss: 0.0014
Epoch 7/10, Loss: 0.0048
Epoch 8/10, Loss: 0.0028
Epoch 9/10, Loss: 0.0009
Epoch 10/10, Loss: 0.0004


In [35]:
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for data, target in test_loader:
        data = data.to(device)
        output = model(data)

        y_true.extend(target.numpy())
        y_pred.extend(output.argmax(1).cpu().numpy())

print("CNN Accuracy:", accuracy_score(y_true, y_pred))
print("F1 micro:", f1_score(y_true, y_pred, average='micro'))
print("F1 macro:", f1_score(y_true, y_pred, average='macro'))

CNN Accuracy: 0.9903
F1 micro: 0.9903
F1 macro: 0.9902051570504738


Сверточная нейросеть оказалась более качественной, поскольку она не теряет пространственную структуру входных изображений